# Ensemble Baseline Models for Maternal Mortality Ratio Prediction

This notebook establishes the baseline performance using three major gradient boosting frameworks: **XGBoost, LightGBM, and CatBoost**.
The pipeline includes data preprocessing, temporal splitting, hyperparameter optimization via Optuna, and performance evaluation with 95% Bootstrap Confidence Intervals.

## 1. Environment Setup & Library Imports
First, we install the required machine learning packages and import necessary libraries.

In [ ]:
# ==========================================
# [1] Required package installation
# ==========================================
!pip install xgboost lightgbm catboost optuna -q

import pandas as pd
import numpy as np
import io
import optuna
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ensemble Model Libraries
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

## 2. Data Loading and Preprocessing
In this section, we load the harmonized dataset.
* **Feature Cleaning:** Column names are sanitized to prevent parsing errors in LightGBM.
* **Missing Values:** Categorical variables are filled with "Unknown", and numerical variables are imputed using the median value.

In [ ]:
# ==========================================
# [2] Data loading and basic preprocessing
# ==========================================
def load_data():
    """
    Load the harmonized dataset for baseline ensemble experiments.
    If 'Dataset.csv' is not found in the directory and the environment is Google Colab,
    it prompts the user to upload the file manually.
    """
    try:
        from google.colab import files
        import os
        if not os.path.exists('Dataset.csv'):
            print("=== [Google Colab] Please upload Dataset.csv ===")
            uploaded = files.upload()
            filename = list(uploaded.keys())[0]
            df = pd.read_csv(io.BytesIO(uploaded[filename]))
        else:
            df = pd.read_csv('Dataset.csv')
    except ImportError:
        # Fallback for local Jupyter environments
        df = pd.read_csv('Dataset.csv')
    return df

df = load_data()

# [Important] Sanitize column names to avoid downstream LightGBM parsing issues
df.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in df.columns]
print("Cleaned columns:", df.columns.tolist())

# Drop redundant identifier column if 'Country_Code' is already available
if 'Country_Name' in df.columns:
    df = df.drop('Country_Name', axis=1)

# Define column groups for modeling
cat_cols = ['Country_Code', 'Continent']
target_col = 'Maternal_Mortality_Ratio'
feature_names = [c for c in df.columns if c not in ['Year', target_col]]

# Handle missing values appropriately based on data types
for col in cat_cols:
    df[col] = df[col].fillna("Unknown") # Fill categorical with placeholder

num_cols = [c for c in df.columns if c not in cat_cols + ['Year', target_col]]
for col in num_cols:
    df[col] = df[col].fillna(df[col].median()) # Impute numerical with median

## 3. Label Encoding & Temporal Data Splitting
Since gradient boosting models handle numerical inputs more efficiently (or require specific categorical handling), we apply Label Encoding.
We design a strict temporal split to prevent data leakage:
* **Phase 1 (Hyperparameter Optimization):** Train on 2011-2014, Validate on 2015.
* **Phase 2 (Retraining):** Train on the full pretest period (2011-2015) using the best hyperparameters.
* **Phase 3 (Final Evaluation):** Test on the held-out year (2016).

In [ ]:
# ==========================================
# [3] Label encoding & Temporal split design
# ==========================================

# Apply Label Encoding to categorical identifiers
df_encoded = df.copy()
for col in cat_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df[col].astype(str))

# Phase 1: Hyperparameter optimization datasets
train_opt = df_encoded[(df_encoded['Year'] >= 2011) & (df_encoded['Year'] <= 2014)]
val_opt   = df_encoded[df_encoded['Year'] == 2015]

# Phase 2: Retraining dataset after model selection
train_retrain = df_encoded[(df_encoded['Year'] >= 2011) & (df_encoded['Year'] <= 2015)]

# Phase 3: Final held-out evaluation dataset
test = df_encoded[df_encoded['Year'] == 2016]

# Utility function to separate features (X) and target (y)
def get_xy(data):
    return data[feature_names], data[target_col]

X_opt_t, y_opt_t = get_xy(train_opt)
X_opt_v, y_opt_v = get_xy(val_opt)
X_final_t, y_final_t = get_xy(train_retrain)
X_test, y_test = get_xy(test)

## 4. Bootstrapping for Confidence Intervals
To ensure the robustness of our evaluation, we utilize a custom function that computes the 95% Bootstrap Confidence Interval for MAE, RMSE, and R-squared metrics.

In [ ]:
# ==========================================
# [4] Bootstrap Confidence Interval Calculation
# ==========================================
def calculate_bootstrap_ci(y_true, y_pred, n_bootstraps=1000, ci=95, random_state=42):
    """
    Calculate the mean performance and confidence intervals using bootstrapping.

    Parameters:
    - y_true: Ground truth target values (1D array)
    - y_pred: Model predictions (1D array)
    - n_bootstraps: Number of resampling iterations (default 1000)
    - ci: Confidence interval percentage (default 95)
    - random_state: Seed for reproducibility

    Returns:
    - Dictionary with formatted strings 'Mean (Lower_Bound - Upper_Bound)' for each metric.
    """
    # Flatten arrays to ensure consistent shapes
    y_true = np.asarray(y_true).flatten()
    y_pred = np.asarray(y_pred).flatten()

    n_samples = len(y_true)
    rng = np.random.RandomState(random_state)

    # Lists to store metrics for each bootstrap iteration
    boot_mae, boot_rmse, boot_r2 = [], [], []

    for _ in range(n_bootstraps):
        # 1. Sample with replacement to create a new bootstrap dataset
        indices = rng.randint(0, n_samples, n_samples)

        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]

        # 2. Calculate and store performance metrics for the sample
        boot_mae.append(mean_absolute_error(y_true_boot, y_pred_boot))
        boot_rmse.append(np.sqrt(mean_squared_error(y_true_boot, y_pred_boot)))
        boot_r2.append(r2_score(y_true_boot, y_pred_boot))

    # 3. Calculate the lower and upper percentiles for the confidence interval
    lower_p = (100 - ci) / 2.0
    upper_p = 100 - lower_p

    results = {}
    metrics = {'MAE': boot_mae, 'RMSE': boot_rmse, 'R2': boot_r2}

    for name, values in metrics.items():
        # Use the mean of the bootstrap distribution
        mean_val = np.mean(values)
        lower_val = np.percentile(values, lower_p)
        upper_val = np.percentile(values, upper_p)

        results[name] = f"{mean_val:.4f} ({lower_val:.4f} - {upper_val:.4f})"

    return results

## 5. Unified Ensemble Training Pipeline
The `run_ensemble` function encapsulates the entire modeling lifecycle for a given algorithm:
1. **Hyperparameter Tuning:** Finds the best parameters using the `Optuna` framework.
2. **Retraining:** Trains the final model on the combined 2011-2015 dataset using the optimized parameters.
3. **Evaluation:** Evaluates predictions on the unseen 2016 test set and generates Bootstrap CIs.
4. **Feature Importance:** Extracts the top drivers of the model's predictions.

In [ ]:
# ==========================================
# [5] Unified ensemble training function
# ==========================================
def run_ensemble(model_name):
    """
    Run a complete baseline experiment for a selected ensemble model.
    """
    print(f"\n{'='*10} Processing {model_name} {'='*10}")

    # --- Step 1: Optuna optimization ---
    def objective(trial):
        # Define hyperparameter search spaces depending on the model
        if model_name == 'XGBoost':
            params = {
                'objective': 'reg:absoluteerror',
                'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
                'max_depth': trial.suggest_int('max_depth', 3, 9),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'n_jobs': -1,
                'eval_metric': 'mae',
                'early_stopping_rounds': 20
            }
            model = xgb.XGBRegressor(**params)
            model.fit(X_opt_t, y_opt_t, eval_set=[(X_opt_v, y_opt_v)], verbose=False)

        elif model_name == 'LightGBM':
            params = {
                'objective': 'regression_l1',
                'n_estimators': trial.suggest_int('n_estimators', 100, 500),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
                'num_leaves': trial.suggest_int('num_leaves', 20, 100),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
                'verbose': -1,
                'n_jobs': -1
            }
            model = lgb.LGBMRegressor(**params)
            # Use LightGBM callback-based early stopping
            model.fit(
                X_opt_t, y_opt_t,
                eval_set=[(X_opt_v, y_opt_v)],
                eval_metric='mae',
                callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
            )

        elif model_name == 'CatBoost':
            params = {
                'loss_function': 'MAE',
                'iterations': trial.suggest_int('iterations', 100, 500),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
                'depth': trial.suggest_int('depth', 4, 10),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
                'verbose': 0,
                'thread_count': -1
            }
            model = CatBoostRegressor(**params)
            cat_features_idx = [i for i, c in enumerate(feature_names) if c in cat_cols]
            model.fit(
                X_opt_t, y_opt_t,
                eval_set=(X_opt_v, y_opt_v),
                cat_features=cat_features_idx,
                early_stopping_rounds=20,
                verbose=0
            )

        # Return MAE on validation set as the optimization metric
        preds = model.predict(X_opt_v)
        return mean_absolute_error(y_opt_v, preds)

    print(">> [Step 1] Optimizing hyperparameters...")
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=10) # Set to 10 for demonstration; increase for production
    best_params = study.best_trial.params
    print(f"   Best parameters: {best_params}")

    # --- Step 2: Retraining on 2011-2015 ---
    print(">> [Step 2] Retraining on the full pretest period (2011-2015)...")

    if model_name == 'XGBoost':
        final_params = best_params.copy()
        # Remove training-only args
        final_params.pop('eval_metric', None)
        final_params.pop('early_stopping_rounds', None)
        final_model = xgb.XGBRegressor(**final_params, n_jobs=-1, enable_categorical=True)

    elif model_name == 'LightGBM':
        final_model = lgb.LGBMRegressor(**best_params, n_jobs=-1, verbose=-1)

    elif model_name == 'CatBoost':
        final_model = CatBoostRegressor(**best_params, verbose=0, thread_count=-1)

    # Fit the final model on the full pretest period
    if model_name == 'CatBoost':
        cat_features_idx = [i for i, c in enumerate(feature_names) if c in cat_cols]
        final_model.fit(X_final_t, y_final_t, cat_features=cat_features_idx, verbose=0)
    else:
        final_model.fit(X_final_t, y_final_t)

    # --- Step 3: Evaluation on the held-out year 2016 ---
    print(">> [Step 3] Evaluating on the held-out test year (2016)...")
    preds = final_model.predict(X_test)

    # Calculate single-point deterministic metrics
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)

    # Calculate robust Bootstrap Confidence Intervals
    boot_ci = calculate_bootstrap_ci(y_test, preds)
    print(f"   [Bootstrap 95% CI] MAE: {boot_ci['MAE']}, RMSE: {boot_ci['RMSE']}, R2: {boot_ci['R2']}")

    # --- Step 4: Extract feature importance ---
    if model_name == 'CatBoost':
        imp_vals = final_model.get_feature_importance()
    elif model_name == 'LightGBM':
        imp_vals = final_model.feature_importances_
    else:  # XGBoost
        imp_vals = final_model.feature_importances_

    importances = pd.Series(imp_vals, index=feature_names).sort_values(ascending=False)

    return {
        'Metrics': {'MAE': mae, 'RMSE': rmse, 'R2': r2},
        'Bootstrap_CI': boot_ci,
        'Importance': importances,
        'Preds': preds
    }

## 6. Execute Baseline Models
Run the pipeline for XGBoost, LightGBM, and CatBoost, storing all results into a dictionary for downstream comparison.

In [ ]:
# ==========================================
# [6] Run all baseline models
# ==========================================
results = {}
models_list = ['XGBoost', 'LightGBM', 'CatBoost']

for m in models_list:
    results[m] = run_ensemble(m)

## 7. Result Comparison and Visualization
Finally, we summarize the model outputs.
* **Tabular Summaries:** Standard metrics vs. 95% Bootstrap CIs.
* **Bar Charts:** Visual comparison of Error (MAE/RMSE) vs. Explanatory Power (R2).
* **Feature Importance:** Top 10 predictive features per model.
* **Scatter Plots:** Actual vs. Predicted values tracking model fit.

In [ ]:
# ==========================================
# [7] Result comparison and visualization
# ==========================================

# 1. Summary Performance Tables
metrics_df = pd.DataFrame(
    {m: results[m]['Metrics'] for m in models_list}
).T[['MAE', 'RMSE', 'R2']]

bootstrap_df = pd.DataFrame(
    {m: results[m]['Bootstrap_CI'] for m in models_list}
).T[['MAE', 'RMSE', 'R2']]

print("\n" + "="*50)
print(" FINAL RESULTS (Held-out Test Year: 2016) ")
print("="*50)
print(metrics_df)

print("\n" + "="*65)
print(" FINAL RESULTS (1000 Bootstraps 95% Confidence Interval) ")
print("="*65)
print(bootstrap_df)

# 2. Performance Comparison Plot (Dual Axis)
fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot Errors (MAE, RMSE) on left Y-axis
metrics_df[['MAE', 'RMSE']].plot(
    kind='bar', ax=ax1, width=0.4, position=1, color=['#74b9ff', '#a29bfe']
)
ax1.set_ylabel("Error (MAE / RMSE)", fontsize=12)
ax1.set_title("Ensemble Baseline Performance Comparison (Held-out Test Year: 2016)", fontsize=14)

# Plot R2 Score on right Y-axis
ax2 = ax1.twinx()
metrics_df['R2'].plot(
    kind='bar', ax=ax2, width=0.2, position=0, color='#ff7675', label='R2 Score'
)
ax2.set_ylabel("R2 Score", color='#ff7675', fontsize=12)
ax2.set_ylim(0, 1.1)

# Fix legend
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2, loc='upper right')

plt.show()

# 3. Feature Importance Comparison (Top 10 Drivers)
fig, axs = plt.subplots(1, 3, figsize=(20, 6))
colors = ['#74b9ff', '#a29bfe', '#ff7675']

for i, model in enumerate(models_list):
    # Extract top 10 and plot horizontally
    results[model]['Importance'].nlargest(10).sort_values().plot(
        kind='barh', ax=axs[i], color=colors[i]
    )
    axs[i].set_title(f"{model} Feature Importance")

plt.tight_layout()
plt.show()

# 4. Actual vs. Predicted Scatter Plots
plt.figure(figsize=(18, 5))
for i, model in enumerate(models_list):
    plt.subplot(1, 3, i + 1)

    # Scatter points for predictions
    sns.scatterplot(
        x=y_test, y=results[model]['Preds'], alpha=0.6, color=colors[i]
    )

    # Ideal prediction line (y = x)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')

    plt.title(f"{model} Predictions (R2: {results[model]['Metrics']['R2']:.3f})")
    plt.xlabel("Actual Values")
    plt.ylabel("Predicted Values")

plt.tight_layout()
plt.show()